# ImageNet ONNX Batch Evaluation → CSV

This notebook:
- loads images from a folder (optionally with class subfolders),
- runs one or more `.onnx` models via **onnxruntime**,
- checks whether prediction matches the ground-truth label inferred from the image path/name,
- saves per-image results to a CSV.


## 1) Config

In [8]:
from pathlib import Path

# === REQUIRED ===
IMAGES_DIR = Path(r"../benchmarks/vggnet16_benchmark2022/imagenet-sample/")
ONNX_PATHS = [Path(r"../benchmarks/vggnet16_benchmark2022/onnx/vgg16-7.onnx")]

# === OUTPUT ===
OUT_CSV = Path("csvs/onnx_imagenet_eval.csv")
# === LABEL INFERENCE MODE ===
# Choose ONE:
# 1) "parent_folder" : expects images organized as IMAGES_DIR/<class_name>/<image>.jpg
# 2) "filename_stem" : expects filename stem is the class label (e.g., "goldfish.jpg")
# 3) "filename_prefix" : expects filename begins with class label + separator (e.g., "goldfish__123.jpg")
LABEL_MODE = "imagenet_filename"  # "filename_stem" | "filename_prefix"

# Only used for filename_prefix mode:
FILENAME_LABEL_SEP = "__"  # e.g. "goldfish__123.jpg" → label "goldfish"

# Optional: limit number of images (set None to use all)
MAX_IMAGES = 1000

# Preprocess settings (ImageNet-style)
RESIZE_SHORTER = 256
CENTER_CROP = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

print("IMAGES_DIR:", IMAGES_DIR)
print("ONNX models:", [str(p) for p in ONNX_PATHS])
print("LABEL_MODE:", LABEL_MODE)
print("OUT_CSV:", OUT_CSV)


IMAGES_DIR: ../benchmarks/vggnet16_benchmark2022/imagenet-sample
ONNX models: ['../benchmarks/vggnet16_benchmark2022/onnx/vgg16-7.onnx']
LABEL_MODE: imagenet_filename
OUT_CSV: csvs/onnx_imagenet_eval.csv


## 2) Utilities (label inference, ImageNet class index, preprocessing)

In [9]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import time
import json
import requests
import re
from pathlib import Path

def list_images(root: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    paths = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]
    paths.sort()
    return paths

def infer_true_label(img_path: Path) -> str:
    if LABEL_MODE == "parent_folder":
        return img_path.parent.name
    elif LABEL_MODE == "filename_stem":
        return img_path.stem
    elif LABEL_MODE == "filename_prefix":
        stem = img_path.stem
        return stem.split(FILENAME_LABEL_SEP)[0]
    else:
        raise ValueError(f"Unknown LABEL_MODE: {LABEL_MODE}")

def download_imagenet_class_index(cache_path: Path = Path("imagenet_class_index.json")):
    """Downloads ImageNet class index mapping: idx -> (wnid, human_readable_label).
    Source: torchvision (commonly used mapping).
    """
    if cache_path.exists():
        with cache_path.open("r") as f:
            return json.load(f)

    url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    # imagenet_classes.txt is 1000 lines, index = line number
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    labels = [line.strip() for line in r.text.splitlines() if line.strip()]
    if len(labels) != 1000:
        raise RuntimeError(f"Expected 1000 ImageNet labels, got {len(labels)}")
    # Build a simple mapping idx -> label
    mapping = {str(i): [None, labels[i]] for i in range(1000)}
    with cache_path.open("w") as f:
        json.dump(mapping, f)
    return mapping

def canonicalize_label(s: str) -> str:
    """Make comparisons forgiving: lowercase, strip, replace spaces/hyphens/underscores."""
    s = str(s).strip().lower()
    s = s.replace("_", " ").replace("-", " ")
    s = " ".join(s.split())
    return s

def infer_true_label(img_path: Path) -> str:
    if LABEL_MODE == "parent_folder":
        return img_path.parent.name

    elif LABEL_MODE == "filename_stem":
        return img_path.stem

    elif LABEL_MODE == "imagenet_filename":
        # Handles: n01440764_tench.JPEG  -> tench
        #          n02123045_tabby_cat.JPEG -> tabby cat (after canonicalize)
        stem = img_path.stem  # no extension
        m = re.match(r"^n\d+_(.+)$", stem)
        if m:
            return m.group(1)
        # fallback: take text after first underscore if present
        if "_" in stem:
            return stem.split("_", 1)[1]
        return stem

    else:
        raise ValueError(f"Unknown LABEL_MODE: {LABEL_MODE}")

def preprocess_imagenet(pil_img: Image.Image) -> np.ndarray:
    """Resize shorter side to RESIZE_SHORTER, center crop CENTER_CROP, normalize, return NCHW float32."""
    img = pil_img.convert("RGB")

    w, h = img.size
    if w < h:
        new_w = RESIZE_SHORTER
        new_h = int(round(h * (RESIZE_SHORTER / w)))
    else:
        new_h = RESIZE_SHORTER
        new_w = int(round(w * (RESIZE_SHORTER / h)))
    img = img.resize((new_w, new_h), Image.BILINEAR)

    # Center crop
    left = (new_w - CENTER_CROP) // 2
    top = (new_h - CENTER_CROP) // 2
    img = img.crop((left, top, left + CENTER_CROP, top + CENTER_CROP))

    x = np.asarray(img).astype(np.float32) / 255.0  # HWC
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std = np.array(IMAGENET_STD, dtype=np.float32)
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1))  # CHW
    x = np.expand_dims(x, axis=0)   # NCHW
    return x

print("Utilities loaded.")


Utilities loaded.


## 3) Load ONNX model(s) with onnxruntime

In [10]:
import onnxruntime as ort

def make_session(onnx_path: Path):
    if not onnx_path.exists():
        raise FileNotFoundError(str(onnx_path))
    # Prefer CUDA if available; fall back to CPU.
    providers = [
        ("CUDAExecutionProvider", {}),
        ("CPUExecutionProvider", {}),
    ]
    # Some installs may not have CUDA provider; ORT will ignore unknown providers.
    sess = ort.InferenceSession(str(onnx_path), providers=[p[0] for p in providers])
    input_name = sess.get_inputs()[0].name
    input_shape = sess.get_inputs()[0].shape
    output_names = [o.name for o in sess.get_outputs()]
    return sess, input_name, input_shape, output_names

sessions = []
for p in ONNX_PATHS:
    sess, in_name, in_shape, out_names = make_session(p)
    sessions.append({
        "model_path": str(p),
        "sess": sess,
        "input_name": in_name,
        "input_shape": in_shape,
        "output_names": out_names,
        "providers": sess.get_providers(),
    })
    print(f"Loaded: {p}")
    print("  providers:", sess.get_providers())
    print("  input:", in_name, in_shape)
    print("  outputs:", out_names)


Loaded: ../benchmarks/vggnet16_benchmark2022/onnx/vgg16-7.onnx
  providers: ['CPUExecutionProvider']
  input: data [1, 3, 224, 224]
  outputs: ['vgg0_dense2_fwd']


## 4) Run evaluation and write CSV

In [11]:
img_paths = list_images(IMAGES_DIR)
if MAX_IMAGES is not None:
    img_paths = img_paths[: int(MAX_IMAGES)]
print(f"Found {len(img_paths)} images")

# Load ImageNet label mapping (idx -> label string)
try:
    class_index = download_imagenet_class_index()
    idx_to_label = {int(k): v[1] for k, v in class_index.items()}
    print("Loaded ImageNet label mapping (1000 classes).")
except Exception as e:
    print("WARNING: Could not load ImageNet label mapping.")
    print("  Error:", e)
    idx_to_label = None

def softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / np.sum(ex)

rows = []

for img_path in tqdm(img_paths, desc="Images"):
    true_label_raw = infer_true_label(img_path)
    true_label = canonicalize_label(true_label_raw)

    try:
        pil = Image.open(img_path)
        x = preprocess_imagenet(pil)
    except Exception as e:
        # record failure for all models
        for m in sessions:
            rows.append({
                "image_path": str(img_path),
                "true_label_raw": true_label_raw,
                "true_label_canon": true_label,
                "model_path": m["model_path"],
                "pred_idx": None,
                "pred_label": None,
                "top5": None,
                "top1_prob": None,
                "correct": False,
                "runtime_ms": None,
                "error": f"image_load/preprocess: {repr(e)}",
            })
        continue

    for m in sessions:
        t0 = time.perf_counter()
        try:
            out = m["sess"].run(None, {m["input_name"]: x})
            # Assume first output contains logits/probs with shape (1,1000) or (1000,)
            y = out[0]
            y = np.array(y)
            if y.ndim == 2:
                y = y[0]
            # Convert logits -> probabilities
            probs = softmax(y.astype(np.float32))
            pred_idx = int(np.argmax(probs))
            top5_idx = np.argsort(-probs)[:5].tolist()
            top1_prob = float(probs[pred_idx])

            if idx_to_label is not None:
                pred_label_raw = idx_to_label.get(pred_idx, str(pred_idx))
                pred_label = canonicalize_label(pred_label_raw)
                top5_labels_raw = [idx_to_label.get(i, str(i)) for i in top5_idx]
                top5_labels = [canonicalize_label(s) for s in top5_labels_raw]
            else:
                pred_label_raw = str(pred_idx)
                pred_label = pred_label_raw
                top5_labels = [str(i) for i in top5_idx]

            correct = (pred_label == true_label)
            runtime_ms = (time.perf_counter() - t0) * 1000.0

            rows.append({
                "image_path": str(img_path),
                "true_label_raw": true_label_raw,
                "true_label_canon": true_label,
                "model_path": m["model_path"],
                "providers": ",".join(m["providers"]),
                "pred_idx": pred_idx,
                "pred_label_raw": pred_label_raw,
                "pred_label_canon": pred_label,
                "top5_idx": top5_idx,
                "top5_labels_canon": top5_labels,
                "top1_prob": top1_prob,
                "correct": bool(correct),
                "runtime_ms": runtime_ms,
                "error": None,
            })

        except Exception as e:
            runtime_ms = (time.perf_counter() - t0) * 1000.0
            rows.append({
                "image_path": str(img_path),
                "true_label_raw": true_label_raw,
                "true_label_canon": true_label,
                "model_path": m["model_path"],
                "providers": ",".join(m["providers"]),
                "pred_idx": None,
                "pred_label_raw": None,
                "pred_label_canon": None,
                "top5_idx": None,
                "top5_labels_canon": None,
                "top1_prob": None,
                "correct": False,
                "runtime_ms": runtime_ms,
                "error": f"onnx_infer: {repr(e)}",
            })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV.resolve())

# Quick summary
summary = (
    df.dropna(subset=["pred_idx"])  # ignore failed inferences
      .groupby("model_path")["correct"]
      .agg(["count", "sum"])
      .rename(columns={"count": "n", "sum": "n_correct"})
)
summary["acc"] = summary["n_correct"] / summary["n"]
display(summary.sort_values("acc", ascending=False))


Found 1000 images
Loaded ImageNet label mapping (1000 classes).


Images: 100%|██████████| 1000/1000 [02:05<00:00,  8.00it/s]

Saved: /Users/zd3504phd/Desktop/XAIV/analysis/csvs/onnx_imagenet_eval.csv


,n,n_correct,acc
model_path,,,
../benchmarks/vggnet16_benchmark2022/onnx/vgg16-7.onnx,1000,868,0.868
